In [103]:
import polars as pl
from polars import col
import numpy as np
import torch
import time

In [18]:
torch.get_default_device()

device(type='cpu')

In [86]:
torch.set_default_device('cuda')

In [87]:
torch.get_default_device()

device(type='cuda', index=0)

In [2]:
df=pl.read_csv('/home/fjc/Documents/data/202302/000001.csv')
df

code,tdate,open,close,high,low,cjl,cje,cjjj
i64,str,f64,f64,f64,f64,f64,f64,f64
1,"""2023-02-01 09:30:00""",15.03,15.03,15.03,15.03,3601.0,5.4123e6,15.03
1,"""2023-02-01 09:31:00""",15.03,14.97,15.08,14.95,19919.0,2.98901e7,15.01
1,"""2023-02-01 09:32:00""",14.97,15.03,15.05,14.97,7600.0,1.14196e7,15.014
1,"""2023-02-01 09:33:00""",15.03,15.02,15.03,15.0,3767.0,5.65412e6,15.013
1,"""2023-02-01 09:34:00""",15.01,15.01,15.02,15.0,5175.0,7.76705e6,15.013
…,…,…,…,…,…,…,…,…
1,"""2023-02-28 14:56:00""",13.78,13.79,13.79,13.77,3833.0,5.28268e6,13.704
1,"""2023-02-28 14:57:00""",13.78,13.79,13.79,13.78,6322.0,8.71505e6,13.704
1,"""2023-02-28 14:58:00""",13.79,13.79,13.79,13.79,812.0,1.11974e6,13.705


In [64]:
sample=df.with_columns(pl.col('open').shift(-2).alias('label')).with_columns([
    ((pl.col(col) - pl.col(col).mean()) / pl.col(col).std()).alias(f"{col}_standardized")
    for col in ['open',	'close',	'high',	'low',	'cjl',	'cje',	'cjjj']
])
sample


code,tdate,open,close,high,low,cjl,cje,cjjj,label,open_standardized,close_standardized,high_standardized,low_standardized,cjl_standardized,cje_standardized,cjjj_standardized
i64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1,"""2023-02-01 09:30:00""",15.03,15.03,15.03,15.03,3601.0,5.4123e6,15.03,14.97,3.722806,3.731986,3.678218,3.775712,-0.180804,-0.128228,3.661964
1,"""2023-02-01 09:31:00""",15.03,14.97,15.08,14.95,19919.0,2.98901e7,15.01,15.03,3.722806,3.515163,3.857909,3.485871,3.102903,3.374274,3.590286
1,"""2023-02-01 09:32:00""",14.97,15.03,15.05,14.97,7600.0,1.14196e7,15.014,15.01,3.506473,3.731986,3.750094,3.558332,0.623924,0.73135,3.604622
1,"""2023-02-01 09:33:00""",15.03,15.02,15.03,15.0,3767.0,5.65412e6,15.013,15.02,3.722806,3.695849,3.678218,3.667022,-0.147399,-0.093627,3.601038
1,"""2023-02-01 09:34:00""",15.01,15.01,15.02,15.0,5175.0,7.76705e6,15.013,15.01,3.650695,3.659712,3.64228,3.667022,0.135936,0.20871,3.601038
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1,"""2023-02-28 14:56:00""",13.78,13.79,13.79,13.77,3833.0,5.28268e6,13.704,13.79,-0.784132,-0.749021,-0.778122,-0.789283,-0.134118,-0.146776,-1.090273
1,"""2023-02-28 14:57:00""",13.78,13.79,13.79,13.78,6322.0,8.71505e6,13.704,13.79,-0.784132,-0.749021,-0.778122,-0.753053,0.366749,0.344358,-1.090273
1,"""2023-02-28 14:58:00""",13.79,13.79,13.79,13.79,812.0,1.11974e6,13.705,13.78,-0.748077,-0.749021,-0.778122,-0.716823,-0.74204,-0.742446,-1.086689


In [65]:
train_data=sample.filter(col('tdate') <= "2023-02-26 15:00:00").select('open_standardized',	'close_standardized',	'high_standardized',	
                                                                       'low_standardized',	'cjl_standardized',	'cje_standardized',
                                                                       'cjjj_standardized','label')
train_X=train_data.drop('label')
train_y=train_data['label']

predict_data=sample.filter(col('tdate') > "2023-02-26 15:00:00").select('open_standardized',	'close_standardized',	'high_standardized',	
                                                                       'low_standardized',	'cjl_standardized',	'cje_standardized',
                                                                       'cjjj_standardized','label')
predict_X=predict_data.drop('label')
predict_y=predict_data['label']

In [88]:
X=train_X.to_torch(dtype=pl.Float32).to('cuda:0', dtype=torch.float)
y=train_y.to_torch().to('cuda:0', dtype=torch.float)
X,y

(tensor([[ 3.7228,  3.7320,  3.6782,  ..., -0.1808, -0.1282,  3.6620],
         [ 3.7228,  3.5152,  3.8579,  ...,  3.1029,  3.3743,  3.5903],
         [ 3.5065,  3.7320,  3.7501,  ...,  0.6239,  0.7313,  3.6046],
         ...,
         [-0.4236, -0.4238, -0.4547,  ..., -0.9004, -0.8977, -0.4882],
         [-0.4236, -0.4238, -0.4547,  ..., -0.9054, -0.9027, -0.4882],
         [-0.4957, -0.4961, -0.5266,  ...,  0.6263,  0.6070, -0.4882]],
        device='cuda:0'),
 tensor([14.9700, 15.0300, 15.0100,  ..., 13.8600, 13.7500, 13.7400],
        device='cuda:0'))

In [77]:
X.dtype

torch.float32

In [89]:
X.device

device(type='cuda', index=0)

# 1. 自行管理迭代和loss
## cpu cost: 48.45271134376526s
## gpu cost: 1.5118281841278076s

In [90]:
# 权重参数初始化.每一层的神经元个数及初始权重.此处共三层
weights = torch.randn((7, 128), dtype = torch.float, requires_grad = True) # 第一层输入7个特征向量(神经元),输出128个特征向量,对应下一层
biases = torch.randn(128, dtype = torch.float, requires_grad = True) 

weights2 = torch.randn((128, 64), dtype = torch.float, requires_grad = True)  #第二层输入128个特征向量(神经元),输出64个特征向量,对应下一层
biases2 = torch.randn(64, dtype = torch.float, requires_grad = True) 

weights3 = torch.randn((64, 1), dtype = torch.float, requires_grad = True) #第二层输入64个特征向量(神经元),输出1个预测值
biases3 = torch.randn(1, dtype = torch.float, requires_grad = True) 

learning_rate = 0.0001  
# 学习率要设得很小,否则梯度爆炸
# relu激活函数允许输出无限大,会导致loss无限大,可以减小学习率,数据归一化等解决,也可以通过梯度裁剪, 切换优化器, 切换激活函数等解决
losses = []

start=time.time()
for i in range(1000):
    # 计算隐层
    hidden = X.mm(weights) + biases
    # 加入激活函数
    hidden = torch.relu(hidden)
    #第二个隐藏层
    hidden = hidden.mm(weights2) + biases2
    hidden = torch.relu(hidden)
    
    # 预测结果
    predictions = hidden.mm(weights3) + biases3
    # 通计算损失
    loss = torch.mean((predictions - y) ** 2) 
    # losses.append(loss.cpu().data.numpy())
    # 打印损失值
    if i % 100 == 0:
        print('loss:', loss)
    #返向传播计算
    loss.backward()  #只计算梯度,不更新参数
    
    #更新参数
    weights.data.add_(- learning_rate * weights.grad.data)  
    biases.data.add_(- learning_rate * biases.grad.data)
    weights2.data.add_(- learning_rate * weights2.grad.data)
    biases2.data.add_(- learning_rate * biases2.grad.data)

    weights3.data.add_(- learning_rate * weights3.grad.data)
    biases3.data.add_(- learning_rate * biases3.grad.data)
    
    # 每次迭代都得记得清空,不然下一次更新参数会使用累计梯度
    weights.grad.data.zero_()
    biases.grad.data.zero_()
    weights2.grad.data.zero_()
    biases2.grad.data.zero_()
    weights3.grad.data.zero_()
    biases3.grad.data.zero_()
end=time.time()
print(f"gpu cost: {end-start}s")

loss: tensor(14337.9668, device='cuda:0', grad_fn=<MeanBackward0>)
loss: tensor(13.0314, device='cuda:0', grad_fn=<MeanBackward0>)
loss: tensor(7.8389, device='cuda:0', grad_fn=<MeanBackward0>)
loss: tensor(5.8750, device='cuda:0', grad_fn=<MeanBackward0>)
loss: tensor(4.7990, device='cuda:0', grad_fn=<MeanBackward0>)
loss: tensor(4.0892, device='cuda:0', grad_fn=<MeanBackward0>)
loss: tensor(3.5702, device='cuda:0', grad_fn=<MeanBackward0>)
loss: tensor(3.1696, device='cuda:0', grad_fn=<MeanBackward0>)
loss: tensor(2.8503, device='cuda:0', grad_fn=<MeanBackward0>)
loss: tensor(2.5914, device='cuda:0', grad_fn=<MeanBackward0>)
gpu cost: 1.5118281841278076s


In [85]:
predictions

tensor([[12.8494],
        [16.5160],
        [17.0770],
        ...,
        [17.3530],
        [17.3295],
        [ 9.0494]], grad_fn=<AddBackward0>)

# 2. 调用torch方法管理网络

In [123]:
input_size = X.shape[1]  #特征数
hidden_size = 128
output_size = 1
batch_size = 256
my_nn = torch.nn.Sequential(
    torch.nn.Linear(input_size, hidden_size),  #输入层
    torch.nn.ReLU(),   # 激活函数
    torch.nn.Linear(hidden_size, output_size), #输出层
)
cost = torch.nn.MSELoss(reduction='mean')  #loss函数
optimizer = torch.optim.Adam(my_nn.parameters(), lr = 0.0001)  #优化器,管理参数更新.可以动态调整学习率,初始较大,然后慢慢变小

In [135]:
# 训练网络
losses = []
for i in range(1000):
    batch_loss = []
    # MINI-Batch方法来进行训练
    for start in range(0, train_X.height, batch_size):
        xx = train_X.slice(start,batch_size).to_torch(dtype=pl.Float32).to('cuda:0', dtype=torch.float)
        yy = train_y.slice(start,batch_size).to_torch().to('cuda:0', dtype=torch.float)
        prediction = my_nn(xx)
        
        loss = cost(prediction.squeeze(), yy)  #prediction输出是(n,1)tensor,需要去除数量为1的维度才能计算loss
        
        optimizer.zero_grad()
        
        loss.backward(retain_graph=True)
        
        optimizer.step()  # 更新参数
        # has_nan = torch.isnan(loss).any()
        # if has_nan:
        #     print(start,batch_size)
        
        batch_loss.append(loss.cpu().data.numpy())
    
    # 打印损失
    if i % 100==0:
        losses.append(np.mean(batch_loss))
        print(i, np.mean(batch_loss))

0 2.835486
100 0.5493558
200 0.07182373
300 0.024019893
400 0.010280422
500 0.004933674
600 0.0031746952
700 0.0032718661
800 0.0027922988
900 0.0024141686


# 预测

In [140]:
predict_tensor = predict_X.to_torch(dtype=pl.Float32).to('cuda:0', dtype=torch.float)
predict = my_nn(predict_tensor).cpu().data.numpy()
predict

array([[13.70953  ],
       [13.783541 ],
       [13.746778 ],
       [13.792823 ],
       [13.7748375],
       [13.750422 ],
       [13.782487 ],
       [13.786427 ],
       [13.791538 ],
       [13.771093 ],
       [13.744836 ],
       [13.757482 ],
       [13.727609 ],
       [13.791236 ],
       [13.787109 ],
       [13.832314 ],
       [13.9065895],
       [13.891856 ],
       [13.841562 ],
       [13.861987 ],
       [13.84472  ],
       [13.92834  ],
       [13.886503 ],
       [13.849197 ],
       [13.876596 ],
       [13.859316 ],
       [13.847126 ],
       [13.85324  ],
       [13.89262  ],
       [13.8511715],
       [13.815617 ],
       [13.794016 ],
       [13.780734 ],
       [13.788962 ],
       [13.786435 ],
       [13.720757 ],
       [13.785664 ],
       [13.8346   ],
       [13.797347 ],
       [13.773494 ],
       [13.802156 ],
       [13.838248 ],
       [13.784607 ],
       [13.794911 ],
       [13.795435 ],
       [13.816881 ],
       [13.78294  ],
       [13.74

In [120]:
train_y.slice(start,batch_size).to_torch().to('cuda:0', dtype=torch.float).squeeze()

tensor([13.8400, 13.8500, 13.8600, 13.8600, 13.8800, 13.9200, 13.9300, 13.9000,
        13.8800, 13.9000, 13.8900, 13.8800, 13.8700, 13.8500, 13.9000, 13.9100,
        13.9300, 13.9500, 13.9600, 13.9800, 13.9800, 14.0200, 14.0300, 14.0200,
        13.9900, 14.0200, 14.0300, 14.0400, 14.0400, 14.0500, 14.0400, 14.0600,
        14.0300, 14.0100, 13.9900, 14.0100, 14.0100, 14.0000, 13.9800, 13.9900,
        13.9800, 14.0200, 14.0200, 14.0200, 14.0500, 14.0300, 14.0200, 14.0200,
        14.0200, 14.0200, 14.0300, 14.0200, 14.0100, 14.0500, 14.0700, 14.0700,
        14.0400, 14.0400, 14.0400, 14.0600, 14.0400, 14.0500, 14.0300, 14.0400,
        14.0400, 14.0100, 14.0000, 13.9700, 13.9600, 13.9500, 13.9700, 14.0000,
        14.0100, 14.0100, 14.0200, 14.0200, 14.0100, 14.0100, 14.0000, 14.0100,
        14.0100, 14.0300, 14.0000, 14.0100, 14.0100, 14.0000, 14.0200, 14.0200,
        14.0500, 14.0600, 14.0600, 14.0500, 14.0400, 14.0400, 14.0500, 14.0500,
        14.0500, 14.0600, 14.0700, 14.07

In [134]:
train_X.height

4338